# PharmaFlow Foundation — Session 2.2 Solutions
### GroupBy, Merge & Pivot Operations

These are the instructor solutions for all six exercise tiers. Every cell below has
been run for real, and the outputs shown are actual output, not illustrative text.

**Three files, all in `data/raw/`:**

| File | Rows | What it is |
|---|---|---|
| `adverse_events_10k.csv` | 6,218 | Session 2.1's dataset: one row per adverse-event report |
| `drug_reference.csv` | 521 | One row per drug that the FDA gives a pharmacologic class. Built from the same openFDA pull (`openfda.pharm_class_epc`, the FDA's "Established Pharmacologic Class"). 604 of the 1,125 drugs have no class in openFDA, so they are not in this table. |
| `reaction_categories.csv` | 287 | Every reaction that appears 5 or more times, grouped into 11 categories. **This grouping was built by hand for this course. It is not an official medical coding.** Rarer reactions are deliberately not in it. |

**pandas version:** these outputs come from pandas 3.0.5. Older pandas prints
`object` where you see `str`; the numbers are the same.


## Before You Run Anything — Environment Check

Same setup pattern as every session: this cell finds the repo no matter which
folder your notebook tool started in, and checks that all three data files are there.


In [1]:
# --- Setup check: always run this cell first ---
import os
import pandas as pd

def find_repo_root(marker_folder="data", start=None):
    """
    Different notebook tools start a notebook's kernel in different folders,
    so we search upward from wherever we happened to start until we find a
    folder that contains `data` -- that only exists at the repo root, so
    wherever we find it IS the repo root.
    """
    current = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(current, marker_folder)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise FileNotFoundError(
                f"Could not find a '{marker_folder}' folder anywhere above "
                f"{os.getcwd()}. Make sure this notebook is somewhere inside "
                f"the pharmaflow-ai repo."
            )
        current = parent


REPO_ROOT = find_repo_root()
RAW = os.path.join(REPO_ROOT, "data", "raw")
EVENTS_FILE = os.path.join(RAW, "adverse_events_10k.csv")
DRUG_REF_FILE = os.path.join(RAW, "drug_reference.csv")
REACTION_FILE = os.path.join(RAW, "reaction_categories.csv")

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 70)

print("pandas version:", pd.__version__)
missing = [p for p in (EVENTS_FILE, DRUG_REF_FILE, REACTION_FILE) if not os.path.exists(p)]
if missing:
    for p in missing:
        print("Missing file:", p)
    print("Fix: put all three CSV files in data/raw/ at the top level of the repo.")
else:
    print("All three data files found in data/raw/")
    print("You're ready to go.")


pandas version: 3.0.5
All three data files found in data/raw/
You're ready to go.


## Warm-Up — A chai stall chain (no FDA data yet)

Nine invented sales rows from four tea stalls, plus a small stall list. Stall S4
sells but is missing from the stall list; stall S5 is on the list but sold nothing.
Every move in this session is practised here first, on numbers you can check by hand.


In [2]:
# WARM-UP: tiny invented data, no FDA records yet.
toy_sales = pd.DataFrame({
    "stall": ["S1", "S1", "S2", "S2", "S3", "S1", "S2", "S3", "S4"],
    "month": ["2026-01", "2026-01", "2026-01", "2026-01", "2026-01",
              "2026-02", "2026-02", "2026-02", "2026-02"],
    "item":  ["Chai", "Coffee", "Chai", "Chai", "Coffee", "Chai", "Coffee", "Chai", "Chai"],
    "cups":  [120, 40, 90, 30, 60, 150, 45, 80, 25],
})
toy_stalls = pd.DataFrame({
    "stall": ["S1", "S2", "S3", "S5"],
    "city":  ["Pune", "Pune", "Indore", "Bhopal"],
})

print("1) cups per stall (one key):")
print(toy_sales.groupby("stall")["cups"].sum())

print()
print("2) cups per stall and month (two keys):")
print(toy_sales.groupby(["stall", "month"])["cups"].sum())

print()
print("3) several answers at once with agg():")
print(toy_sales.groupby("stall").agg(rows=("cups", "count"), total_cups=("cups", "sum"),
                                     biggest_sale=("cups", "max")))

print()
print("4) rows after each kind of merge:")
for how in ["inner", "left", "right", "outer"]:
    print(f"   {how:5}:", len(toy_sales.merge(toy_stalls, on="stall", how=how)))

print()
print("5) month x stall table with pivot_table:")
monthly = toy_sales.pivot_table(index="month", columns="stall", values="cups",
                                aggfunc="sum", fill_value=0)
print(monthly)

print()
print("6) change from January to February (pct_change) and running total (cumsum):")
print(monthly.pct_change().round(2))
print(monthly.cumsum())


1) cups per stall (one key):
stall
S1    310
S2    165
S3    140
S4     25
Name: cups, dtype: int64

2) cups per stall and month (two keys):
stall  month  
S1     2026-01    160
       2026-02    150
S2     2026-01    120
       2026-02     45
S3     2026-01     60
       2026-02     80
S4     2026-02     25
Name: cups, dtype: int64

3) several answers at once with agg():
       rows  total_cups  biggest_sale
stall                                
S1        3         310           150
S2        3         165            90
S3        2         140            80
S4        1          25            25

4) rows after each kind of merge:
   inner: 8
   left : 9
   right: 9
   outer: 10

5) month x stall table with pivot_table:
stall     S1   S2  S3  S4
month                    
2026-01  160  120  60   0
2026-02  150   45  80  25

6) change from January to February (pct_change) and running total (cumsum):
stall      S1    S2    S3   S4
month                         
2026-01   NaN   NaN   NaN  N

## Exercise 1 · Beginner — "Which drugs have the highest share of serious reports?"

The Pharmacovigilance Lead wants, for every drug with **at least 30 reports**: how
many reports it has, how many are serious, and the share that are serious. A share
from 3 reports means nothing, which is why the threshold exists. She also wants
the same split by country for the five countries that send the most reports.


In [3]:
# Exercise 1: groupby with one key, agg with named outputs, then two keys.
TEXT_COLUMNS = {"safety_report_id": str, "serious": str,
                "patient_age_unit": str, "country": str, "event_date": str}
df = pd.read_csv(EVENTS_FILE, dtype=TEXT_COLUMNS)

# 1.0 / 0.0 for serious / not serious; reports with no value stay missing.
df["is_serious"] = df["serious"].map({"1": 1.0, "2": 0.0})

print("rows per drug, size() vs count():")
print("  size()  counts every row:           ", df.groupby("drug_name").size().sum())
print("  count() skips missing 'serious':    ", df.groupby("drug_name")["serious"].count().sum())

by_drug = df.groupby("drug_name").agg(
    reports=("safety_report_id", "count"),
    serious_reports=("is_serious", "sum"),
    serious_rate=("is_serious", "mean"),
)
print()
print("drugs in the table:", len(by_drug))
print("drugs with at least 30 reports:", (by_drug["reports"] >= 30).sum())

busy = by_drug[by_drug["reports"] >= 30].sort_values("serious_rate", ascending=False)
print()
print("highest serious share (30+ reports):")
print(busy.head(5).round(3))
print()
print("lowest serious share (30+ reports):")
print(busy.tail(3).round(3))


rows per drug, size() vs count():
  size()  counts every row:            6218
  count() skips missing 'serious':     6205

drugs in the table: 1125
drugs with at least 30 reports: 32

highest serious share (30+ reports):
               reports  serious_reports  serious_rate
drug_name                                            
DAPAGLIFLOZIN       34             34.0         1.000
OSIMERTINIB         32             32.0         1.000
VENCLEXTA           32             32.0         1.000
CLOZAPINE           35             34.0         0.971
SKYRIZI            139            124.0         0.892

lowest serious share (30+ reports):
                          reports  serious_reports  serious_rate
drug_name                                                       
OXERVATE                       70              0.0           0.0
POLYETHYLENE GLYCOL 3350       68              0.0           0.0
RYALTRIS                       79              0.0           0.0


In [4]:
# The same table written with a dictionary, agg({...}). Same numbers, but the
# columns keep their old names, which is why the named style above is easier to read.
print(df.groupby("drug_name").agg({"safety_report_id": "count", "is_serious": "mean"})
        .loc[["DUPIXENT", "CABOMETYX"]].round(3))

# Two keys: country and seriousness, for the five busiest countries.
df["country_display"] = df["country"].fillna("Not reported")
df["serious_label"] = df["serious"].map({"1": "Serious", "2": "Not serious"}).fillna("Not recorded")

top5 = df["country_display"].value_counts().head(5).index
by_country = (df[df["country_display"].isin(top5)]
              .groupby(["country_display", "serious_label"], as_index=False)
              .agg(reports=("safety_report_id", "count")))
print()
print(by_country)


           safety_report_id  is_serious
drug_name                              
DUPIXENT                621       0.084
CABOMETYX               170       0.100

   country_display serious_label  reports
0               EU   Not serious       18
1               EU       Serious      211
2               GB   Not serious        6
3               GB       Serious      247
4               JP   Not serious       25
5               JP       Serious      161
6     Not reported   Not serious      174
7     Not reported       Serious      229
8               US  Not recorded       12
9               US   Not serious     2984
10              US       Serious     1238


## Exercise 2 · Intermediate — "Report by drug class, but tell me what we can't classify"

Medical Affairs wants reports and the serious share **by drug class**, not by
product. The class lives in a second file, `drug_reference.csv`. Before trusting
any class-level number, find out how many reports actually get a class, and which
big drugs are missing from the reference table.


In [5]:
# Exercise 2: merge the drug reference table and measure what each join keeps.
drug_ref = pd.read_csv(DRUG_REF_FILE)
print("reference table:", drug_ref.shape, "| drug_name unique:", drug_ref["drug_name"].is_unique)
print(drug_ref.head(3))

print()
print("rows kept by each join type (events has", len(df), "rows):")
for how in ["inner", "left", "right", "outer"]:
    print(f"   {how:5}:", len(df.merge(drug_ref, on="drug_name", how=how)))

classified = df.merge(drug_ref, on="drug_name", how="left",
                      validate="many_to_one", indicator=True)
print()
print("where each row came from (_merge):")
print(classified["_merge"].value_counts())


reference table: (521, 3) | drug_name unique: True
       drug_name                          drug_class  n_classes
0       ABRAXANE               Microtubule Inhibitor          1
1         ABREVA  Non-Standardized Chemical Allergen          1
2  ACALABRUTINIB                    Kinase Inhibitor          1

rows kept by each join type (events has 6218 rows):
   inner: 3987
   left : 6218
   right: 3987
   outer: 6218

where each row came from (_merge):
_merge
both          3987
left_only     2231
right_only       0
Name: count, dtype: int64


In [6]:
# What an inner join would silently throw away.
no_class = classified[classified["_merge"] == "left_only"]
print("reports with no drug class:", len(no_class),
      f"({len(no_class) / len(classified):.1%} of all reports)")
print()
print("biggest drugs missing from the reference table:")
print(no_class["drug_name"].value_counts().head(5))

classified["drug_class"] = classified["drug_class"].fillna("Unclassified")
by_class = classified.groupby("drug_class").agg(
    reports=("safety_report_id", "count"),
    drugs=("drug_name", "nunique"),
    serious_rate=("is_serious", "mean"),
).sort_values("reports", ascending=False)
print()
print("top 6 rows of the class table:")
print(by_class.head(6).round(3))

top_class = by_class.index[1]
in_top_class = classified[classified["drug_class"] == top_class]
print()
print(f"inside '{top_class}':")
print(in_top_class["drug_name"].value_counts())


reports with no drug class: 2231 (35.9% of all reports)

biggest drugs missing from the reference table:
drug_name
CABOMETYX    170
RYALTRIS      79
HUMIRA        71
CREON         55
NUPLAZID      55
Name: count, dtype: int64

top 6 rows of the class table:
                                                               reports  drugs  serious_rate
drug_class                                                                                 
Unclassified                                                      2231    604         0.528
Interleukin-4 Receptor alpha Antagonist                            630      2         0.092
Interleukin-23 Antagonist                                          171      5         0.754
Thalidomide Analog                                                 161      4         0.446
Atypical Antipsychotic                                             148     11         0.568
Glucose-dependent Insulinotropic Polypeptide Receptor Agonist      132      4         0.295

insid

In [7]:
# Why validate= matters: a reference table with one drug listed twice.
broken_ref = pd.concat([drug_ref, drug_ref[drug_ref["drug_name"] == "DUPIXENT"]])
print("rows after merging with the broken reference:", len(df.merge(broken_ref, on="drug_name", how="left")))
try:
    df.merge(broken_ref, on="drug_name", how="left", validate="many_to_one")
except pd.errors.MergeError as err:
    print("validate='many_to_one' stops it:", str(err).splitlines()[0])


rows after merging with the broken reference: 6839
validate='many_to_one' stops it: Merge keys are not unique in right dataset; not a many-to-one merge


## Exercise 3 · Advanced — "Which kinds of reaction are most often serious?"

The Pharmacovigilance Lead wants a table with one row per reaction category and
columns for Serious, Not serious and Not recorded, plus each category's serious
share. Build it twice (with `pivot_table` and with `crosstab`) and prove the two
agree, then reshape it between wide and long form with `stack` and `unstack`.


In [8]:
# Exercise 3: attach categories, check coverage, then pivot_table vs crosstab.
reaction_map = pd.read_csv(REACTION_FILE)
print("lookup:", reaction_map.shape, "| categories:", reaction_map["reaction_category"].nunique())

rx = df.merge(reaction_map, on="reaction", how="left", validate="many_to_one")
mapped = rx["reaction_category"].notna()
print("reports with a category:", mapped.sum(), f"({mapped.mean():.1%})")
print("reports left unmapped:  ", (~mapped).sum())
print()
print("serious share, mapped vs unmapped reports:")
print(rx.groupby(mapped.map({True: "mapped", False: "unmapped"}))["is_serious"].mean().round(3))

rx["reaction_category"] = rx["reaction_category"].fillna("Unmapped")


lookup: (287, 2) | categories: 11
reports with a category: 4283 (68.9%)
reports left unmapped:   1935

serious share, mapped vs unmapped reports:
reaction_category
mapped      0.397
unmapped    0.633
Name: is_serious, dtype: float64


In [9]:
counts_pt = rx.pivot_table(index="reaction_category", columns="serious_label",
                           values="safety_report_id", aggfunc="count", fill_value=0)
counts_ct = pd.crosstab(rx["reaction_category"], rx["serious_label"])
print("pivot_table and crosstab give the same counts:", counts_pt.equals(counts_ct))

table = counts_ct[["Serious", "Not serious", "Not recorded"]].copy()
table["serious_share"] = (table["Serious"] / (table["Serious"] + table["Not serious"])).round(3)
table = table.sort_values("serious_share", ascending=False)
print()
print(table)

print()
print("same shares straight from crosstab(normalize='index'), recorded rows only:")
recorded = rx[rx["serious_label"] != "Not recorded"]
print(pd.crosstab(recorded["reaction_category"], recorded["serious_label"], normalize="index")
        .round(3).sort_values("Serious", ascending=False).head(3))


pivot_table and crosstab give the same counts: True

serious_label                     Serious  Not serious  Not recorded  serious_share
reaction_category                                                                  
Death or hospital care                377           33             0          0.920
Heart, blood and circulation          241           79             0          0.753
Unmapped                             1222          709             4          0.633
Other organs and cancer               188          140             0          0.573
Muscles, bones and joints             113          103             0          0.523
Brain, nerves and mood                195          184             1          0.515
Infection and breathing               165          158             1          0.511
Gut and liver                         139          186             0          0.428
Lack of effect                         80          231             1          0.257
General symptoms       

In [10]:
# Wide -> long -> wide.
long_form = counts_ct.stack()
print("wide shape:", counts_ct.shape, "-> long length:", len(long_form))
print(long_form.head(4))
back = long_form.unstack()
print()
print("unstack() rebuilds the wide table exactly:", back.equals(counts_ct))


wide shape: (12, 3) -> long length: 36
reaction_category       serious_label
Brain, nerves and mood  Not recorded       1
                        Not serious      184
                        Serious          195
Death or hospital care  Not recorded       0
dtype: int64

unstack() rebuilds the wide table exactly: True


## Exercise 4 · Corporate Scenario — "DUPIXENT tripled. Is it a safety signal?"

Regulatory Affairs has spotted that DUPIXENT reports jumped in 2026 Q1 and asks
whether the company must investigate a new safety signal. Build quarterly counts
for the five most-reported drugs, compute the quarter-over-quarter change correctly
(never across two drugs, never skipping a quarter with zero reports), add a running
total, and then check whether the *kind* of report changed or only the number.

One fact about this dataset matters here: the pull took the same number of raw
reports from every quarter (1,250), so a quarter's total says nothing about real
national volume. A **share** of the quarter is the fair comparison.


In [11]:
# Exercise 4: quarterly counts, done right.
df["quarter"] = pd.to_datetime(df["event_date"], format="%Y%m%d").dt.to_period("Q").astype(str)
top5_drugs = df["drug_name"].value_counts().head(5).index
print("five most-reported drugs:", list(top5_drugs))

long_counts = df[df["drug_name"].isin(top5_drugs)].groupby(["drug_name", "quarter"]).size()
print("rows in the long table:", len(long_counts), "(5 drugs x 8 quarters would be 40)")

print()
print("WRONG: pct_change on the long table runs straight across drug boundaries:")
print(long_counts.pct_change().loc[["CABOMETYX", "DUPIXENT"]].head(5).round(3))

quarterly = long_counts.unstack("drug_name", fill_value=0)
print()
print("RIGHT: quarters as rows, one column per drug, zeros filled in:")
print(quarterly)


five most-reported drugs: ['DUPIXENT', 'CABOMETYX', 'SKYRIZI', 'RINVOQ', 'RYALTRIS']
rows in the long table: 28 (5 drugs x 8 quarters would be 40)

WRONG: pct_change on the long table runs straight across drug boundaries:
drug_name  quarter
CABOMETYX  2025Q1        NaN
           2025Q3     -0.988
           2026Q1     84.000
DUPIXENT   2024Q3     -0.753
           2024Q4      3.190
dtype: float64

RIGHT: quarters as rows, one column per drug, zeros filled in:
drug_name  CABOMETYX  DUPIXENT  RINVOQ  RYALTRIS  SKYRIZI
quarter                                                  
2024Q3             0        21      10         0        5
2024Q4             0        88      17         0       14
2025Q1            84        24      11        79       18
2025Q2             0        52      17         0       22
2025Q3             1        44      14         0       27
2025Q4             0        45       8         0       15
2026Q1            85       180       5         0        8
2026Q2       

In [12]:
qoq = quarterly.pct_change()
print("quarter-over-quarter change for DUPIXENT and SKYRIZI:")
print(qoq[["DUPIXENT", "SKYRIZI"]].round(3))
print()
print("CABOMETYX shows why 0 -> n needs care (inf = growth from nothing):")
print(qoq["CABOMETYX"].head(4))

print()
print("running total of DUPIXENT reports (cumsum):")
print(quarterly["DUPIXENT"].cumsum().to_dict())

share = quarterly["DUPIXENT"] / df.groupby("quarter").size()
print()
print("DUPIXENT as a share of each quarter's reports:")
print(share.round(3).to_dict())


quarter-over-quarter change for DUPIXENT and SKYRIZI:
drug_name  DUPIXENT  SKYRIZI
quarter                     
2024Q3          NaN      NaN
2024Q4        3.190    1.800
2025Q1       -0.727    0.286
2025Q2        1.167    0.222
2025Q3       -0.154    0.227
2025Q4        0.023   -0.444
2026Q1        3.000   -0.467
2026Q2       -0.072    2.750

CABOMETYX shows why 0 -> n needs care (inf = growth from nothing):
quarter
2024Q3    NaN
2024Q4    NaN
2025Q1    inf
2025Q2   -1.0
Name: CABOMETYX, dtype: float64

running total of DUPIXENT reports (cumsum):
{'2024Q3': 21, '2024Q4': 109, '2025Q1': 133, '2025Q2': 185, '2025Q3': 229, '2025Q4': 274, '2026Q1': 454, '2026Q2': 621}

DUPIXENT as a share of each quarter's reports:
{'2024Q3': 0.028, '2024Q4': 0.119, '2025Q1': 0.03, '2025Q2': 0.067, '2025Q3': 0.06, '2025Q4': 0.059, '2026Q1': 0.216, '2026Q2': 0.203}


In [13]:
# Did the KIND of DUPIXENT report change, or only the number?
dupi = df[df["drug_name"] == "DUPIXENT"].merge(reaction_map, on="reaction", how="left", validate="many_to_one")
dupi["reaction_category"] = dupi["reaction_category"].fillna("Unmapped")
dupi["period"] = dupi["quarter"].isin(["2026Q1", "2026Q2"]).map({True: "2026 H1", False: "Before 2026"})

mix = pd.crosstab(dupi["reaction_category"], dupi["period"], normalize="columns").round(3)
print("reaction mix (share of each period's DUPIXENT reports):")
print(mix.sort_values("2026 H1", ascending=False).head(5))

print()
print(dupi.groupby("period").agg(reports=("safety_report_id", "count"),
                                 serious_rate=("is_serious", "mean")).round(3))
print()
print("2026 Q1 DUPIXENT reports by receive date (top 3):")
print(dupi.loc[dupi["quarter"] == "2026Q1", "event_date"].value_counts().head(3))


reaction mix (share of each period's DUPIXENT reports):
period                            2026 H1  Before 2026
reaction_category                                     
Skin, allergy and injection site    0.363        0.328
Unmapped                            0.184        0.252
Product use or dosing error         0.112        0.109
Infection and breathing             0.089        0.091
General symptoms                    0.061        0.044

             reports  serious_rate
period                            
2026 H1          347         0.052
Before 2026      274         0.124

2026 Q1 DUPIXENT reports by receive date (top 3):
event_date
20260108    77
20260104    36
20260301    32
Name: count, dtype: int64


## Exercise 5 · Interview Challenge — A quarter-over-quarter function that never lies

Write `qoq_change(frame, by)`. `frame` has a `quarter` column and a column named
by `by`. Return a table with one row per quarter and one column per value of `by`,
holding the **percent** change in report counts from the previous quarter, rounded
to 1 decimal. Rules: no `for` or `while` loops; a quarter with no reports for a
group counts as 0, not as missing; growth from 0 is shown as missing (NaN), not
`inf`; it must not change `frame`; and it must work on a DataFrame with zero rows.


In [14]:
# Exercise 5: vectorised, zero-filled, inf-free, side-effect free.
import numpy as np

def qoq_change(frame, by):
    """Percent change in row counts per quarter, one column per value of `by`."""
    counts = frame.groupby(["quarter", by]).size().unstack(by, fill_value=0)
    change = counts.pct_change() * 100
    return change.replace([np.inf, -np.inf], np.nan).round(1)


before = df.copy()
big4 = df["country_display"].value_counts().head(4).index
print(qoq_change(df[df["country_display"].isin(big4)], "country_display"))
print()
print("input unchanged:", df.equals(before))
print()
print("empty DataFrame:")
print(qoq_change(df.iloc[0:0], "country_display"))
print()
toy = toy_sales.rename(columns={"month": "quarter"})
print("toy check (stalls, month as the period):")
print(qoq_change(toy, "stall"))


country_display     EU    GB  Not reported    US
quarter                                         
2024Q3             NaN   NaN           NaN   NaN
2024Q4             0.0 -39.5         -37.3   2.9
2025Q1            33.3  21.7           6.2  14.3
2025Q2           300.0  17.9         144.1 -14.1
2025Q3           187.5  18.2         -67.5   5.7
2025Q4            30.4 -12.8         148.1  -0.2
2026Q1           -25.0 -29.4         -64.2  25.7
2026Q2            15.6  41.7         254.2 -18.4

input unchanged: True

empty DataFrame:
Empty DataFrame
Columns: []
Index: []

toy check (stalls, month as the period):
stall      S1    S2   S3  S4
quarter                     
2026-01   NaN   NaN  NaN NaN
2026-02 -50.0 -50.0  0.0 NaN


---
### Grading notes

- **Warm-Up:** cups per stall S1 310, S2 165, S3 140, S4 25. Merges keep inner 8 (the
  S4 sale has no stall-list match and is dropped), left 9 (S4 kept, city blank), right 9
  (8 matched sales plus S5 with no sales) and outer 10. Without `fill_value=0` the pivot
  shows S4's January as NaN, not 0. February's change for S4 is `inf`: growth from 0 has
  no percentage, which sets up Exercise 5.
- **Exercise 1:** `size()` counts 6,218 rows; `count()` on `serious` counts 6,205, because
  13 reports have no seriousness recorded. `mean()` of the 1.0/0.0 column skips those 13,
  so the serious share is over recorded reports only; the answer should say so. 32 drugs
  have 30+ reports. Top: DAPAGLIFLOZIN, OSIMERTINIB and VENCLEXTA at 100%, CLOZAPINE 97.1%,
  SKYRIZI 89.2% (124 of 139). Bottom: OXERVATE, POLYETHYLENE GLYCOL 3350 and RYALTRIS at 0%.
  Each of those three is a single-day US batch (70, 68 and 79 reports received on one date
  each), so "0% serious" describes one submission, not the medicine. Two-key table: almost
  every GB report is serious (247 of 253), while most US reports are not (1,238 serious of
  4,234). A student who concludes that UK patients have worse reactions has missed that the
  countries send different kinds of report.
- **Exercise 2:** inner 3,987, left 6,218, right 3,987, outer 6,218. Right equals inner and
  outer equals left because every drug in the reference table appears in the events file
  (`right_only` is 0). 2,231 reports (35.9%) have no class; the biggest are CABOMETYX 170,
  RYALTRIS 79, HUMIRA 71, CREON 55 and NUPLAZID 55. An inner join would silently drop all of
  them. "Unclassified" is the largest row of the class table (52.8% serious), so it must be
  reported, not hidden. The second row, Interleukin-4 Receptor alpha Antagonist (630
  reports), is DUPIXENT 621 plus DUPILUMAB 9: the same medicine under its brand name and
  its generic name. Class-level numbers here are essentially DUPIXENT's numbers. The broken
  reference grows the result from 6,218 to 6,839 rows (621 DUPIXENT rows doubled) with no
  error, and `validate="many_to_one"` stops it with a `MergeError`.
- **Exercise 3:** 4,283 reports (68.9%) get a category and 1,935 stay unmapped. Unmapped
  reports are serious 63.3% of the time against 39.7% for mapped ones: rare reactions tend
  to be serious, so dropping "Unmapped" would understate risk. `pivot_table` and `crosstab`
  agree exactly. Highest serious share: Death or hospital care 92.0% (not 100%: 33 of these
  reports are coded not serious, a data-quality question for the safety team). Lowest:
  Product use or dosing error 4.9%. The serious share excludes "Not recorded" from the
  denominator; `normalize="index"` on recorded rows gives the same 0.920. `stack()` turns
  the 12 x 3 table into 36 rows and `unstack()` rebuilds it exactly.
- **Exercise 4:** only 28 drug-quarter rows exist out of 40, because `groupby().size()`
  leaves out quarters with no reports. On the long table, DUPIXENT's first quarter shows
  -75.3%: that is 21 compared with CABOMETYX's last value, 85, a different drug. After
  `unstack(fill_value=0)`, DUPIXENT 2026 Q1 is +300% (45 to 180). CABOMETYX shows `inf` in
  2025 Q1 (0 to 84). As a share of each quarter, DUPIXENT goes from 5.9% to 21.6%. But the
  reaction mix barely moves (skin, allergy and injection site 32.8% before 2026 vs 36.3% in
  2026 H1), and the serious share falls from 12.4% to 5.2%. The expected business answer:
  more DUPIXENT reports, not a new kind of report, so this data does not show a new safety
  signal. Recommend confirming against full openFDA counts, because this sample took a
  fixed number of reports per quarter. 77 of the 180 Q1 reports share one receive date
  (2026-01-08).
- **Exercise 5:** check for loops, for `inplace=True`, and that the empty case prints an
  empty table without raising. On the toy data the function counts rows, not cups: S1 and
  S2 go from 2 rows to 1 (-50%), S3 stays at 1 (0%), and S4 is NaN (growth from 0), not
  `inf`. A solution that fills NaN with 0 is wrong: 0% would claim "no change" for a stall
  that did not exist in January.
